# Tetrahedral mesher backends compared

`tetrahedralize` fills a closed triangle surface with tetrahedra, and it reaches several meshers by name: the native **octree** backend (always built), **Netgen**, and the installed **TetGen** and **Gmsh** programs. The octree backend also has an opt-in remeshing pass from v20.1.0, `post_adapt=True` (`adapt_volume` run on its tetrahedra), so this notebook compares it twice: plain, and **adapted**.

Each backend meshes the same four skins, three primitives and one scanned model (the Stanford bunny), at a target edge `h` per skin. Besides the number of tetrahedra the comparison reports what the mesh is worth: whether the skin is kept, the distance to it, volume error, the dihedral-angle range, sliver counts, and the distributions of the scaled Jacobian and aspect ratio from `compute_quality`. See `doc/tetrahedralize.md` ("Backend comparison") for the tables this notebook produced.

In [ ]:
import time
import warnings

import numpy as np
import matplotlib.pyplot as plt

import meshioplusplus as mp

try:
    import pyvista as pv
    pv.OFF_SCREEN = True
    pv.set_jupyter_backend('static')
except ImportError:  # pragma: no cover
    pv = None


def gl_available():
    """Probe off-screen GL in a child process: a missing GL stack can kill the
    interpreter outright, which no try/except in this one would survive."""
    if pv is None:
        return False
    import subprocess, sys
    probe = ('import pyvista as pv; pv.OFF_SCREEN = True; pl = pv.Plotter(off_screen=True); '
             'pl.add_mesh(pv.Sphere()); pl.screenshot(return_img=True)')
    try:
        return subprocess.run([sys.executable, '-c', probe], capture_output=True,
                              timeout=120).returncode == 0
    except Exception:
        return False


GL = gl_available()
print('PyVista rendering' if GL else 'matplotlib fallback (no off-screen GL)')

warnings.simplefilter('ignore', RuntimeWarning)
print('backends:', mp.tetrahedralize_backends(), '| netgen built in:', mp.has_netgen())

## The skins

A sphere (curved, no sharp edges), a unit cube (flat faces and 90-degree edges, which a fitted boundary must reproduce), a cylinder (both), and a **complex geometry**: the Stanford bunny scan shipped in `example/data` (112,402 triangles, thin ears, concave folds), remeshed to 3,000 triangles with `remesh` (ACVD clustering) and orientation-repaired with `repair`, since a mesher needs a closed, consistently oriented skin. The cube and cylinder skins are the boundaries of built-in grids (the cylinder's hexahedra split into tetrahedra by `convert_cells`).

In [ ]:
import os
BUNNY = os.path.join(os.path.dirname(os.getcwd()), 'data', 'Bunny.stl')
bunny = mp.repair(mp.remesh(mp.read(BUNNY), 1500))
bb = bunny.points.max(axis=0) - bunny.points.min(axis=0)
H = {'sphere': 0.2, 'cube': 0.2, 'cylinder': 0.2, 'bunny': float(bb.max()) / 18}
SKINS = {
    'sphere': (mp.icosa_sphere(1.0, 3), 4.0 / 3.0 * np.pi),
    'cube': (mp.extract_skin(mp.box((5, 5, 5), (1, 1, 1), cell_type='tetra')), 1.0),
    'cylinder': (mp.extract_skin(mp.convert_cells(mp.cylinder(0.5, 1.0, nxy=10, nz=5), mode='simplexify')), np.pi * 0.25),
    'bunny': (bunny, abs(mp.compute_stats(bunny)['signed_volume'])),
}
for name, (s, v) in SKINS.items():
    print(f'{name:9s} {sum(len(c.data) for c in s.cells)} triangles, h = {H[name]:.3f}, enclosed volume {v:.4f}')

## One function per backend

Every backend returns a mesh of tetrahedra and a report. `octree + adapt` calls `octree_mesh(..., post_adapt=True)` with `adapt_hmax = h` and `adapt_max_iterations=3`; the plain octree row is `tetrahedralize(backend='octree')`, the same stuffing without the remeshing pass. Quality is measured afterwards on the tetrahedra alone with `compute_quality`, identically for every backend.

In [ ]:
def tets_of(mesh):
    blocks = [c for c in mesh.cells if c.type == 'tetra']
    return mp.Mesh(mesh.points, [(b.type, b.data) for b in blocks])


def run_tetrahedralize(name):
    def run(skin, h):
        vol, rep = mp.tetrahedralize(skin, backend=name, max_size=h, return_report=True)
        return vol, rep.get('boundary_preserved'), rep.get('hausdorff_to_surface')
    return run


def run_octree_adapt(skin, h):
    # adapt_max_iterations=3 bounds the (slow, see the timings) remeshing pass
    vol, rep = mp.octree_mesh(skin, output='tet', max_size=h, post_adapt=True,
                              adapt_hmax=h, adapt_max_iterations=3, return_report=True)
    return vol, False, rep.get('hausdorff_to_skin')


RUNNERS = {}
for name in mp.tetrahedralize_backends():
    RUNNERS[name] = run_tetrahedralize(name)
RUNNERS['octree + adapt'] = run_octree_adapt
order = [n for n in ('octree', 'octree + adapt', 'netgen', 'tetgen', 'gmsh') if n in RUNNERS]
print(order)

In [ ]:
def dihedrals_ok(q, key):
    m = q['metrics'][key]
    return m['min'], m['max'], m['mean']


RESULTS = {}   # (skin, backend) -> dict
for sname, (skin, true_vol) in SKINS.items():
    for bname in order:
        best, out = None, None
        for _ in range(1 if bname == 'octree + adapt' else 3):
            t0 = time.perf_counter()
            out = RUNNERS[bname](skin, H[sname])
            dt = time.perf_counter() - t0
            best = dt if best is None else min(best, dt)
        vol, kept, hd = out
        tets = tets_of(vol)
        q = mp.compute_quality(tets)
        cell = q['cell_arrays']
        sj = np.concatenate([np.ravel(a) for a in cell['quality:scaled_jacobian']])
        ar = np.concatenate([np.ravel(a) for a in cell['quality:aspect_ratio']])
        dmin = np.concatenate([np.ravel(a) for a in cell['quality:min_dihedral']])
        dmax = np.concatenate([np.ravel(a) for a in cell['quality:max_dihedral']])
        v = np.concatenate([np.ravel(a) for a in cell['quality:volume']])
        RESULTS[(sname, bname)] = dict(
            tets=len(sj), points=len(tets.points), time=best, kept=bool(kept), hausdorff=hd,
            vol_err=100.0 * (v.sum() - true_vol) / true_vol,
            dmin=dmin.min(), dmax=dmax.max(),
            sliver15=int((dmin < 15).sum()), sliver_sj=int((sj < 0.2).sum()),
            sj_min=sj.min(), sj_mean=sj.mean(), ar_max=ar.max(), ar_mean=ar.mean(),
            inverted=q['num_inverted'], degenerate=q['num_degenerate'],
            sj=sj, ar=ar, dmin_all=dmin, dmax_all=dmax, mesh=tets)
print('done', len(RESULTS), 'runs')

## Results

The tables below are what `doc/tetrahedralize.md` quotes.

In [ ]:
cols = [('tets', 'tets', '{:d}'), ('points', 'points', '{:d}'), ('time', 'time s', '{:.3f}'),
        ('kept', 'skin kept', '{}'), ('hausdorff', 'dist. to skin', '{:.4f}'),
        ('vol_err', 'volume err %', '{:+.2f}'),
        ('dmin', 'min dihedral', '{:.1f}'), ('dmax', 'max dihedral', '{:.1f}'),
        ('sliver15', 'dihedral<15', '{:d}'), ('sliver_sj', 'SJ<0.2', '{:d}'),
        ('sj_min', 'SJ min', '{:.3f}'), ('sj_mean', 'SJ mean', '{:.3f}'),
        ('ar_max', 'AR max', '{:.2f}'), ('ar_mean', 'AR mean', '{:.2f}'),
        ('inverted', 'inverted', '{:d}')]


def table(sname):
    head = ['backend'] + [c[1] for c in cols]
    rows = []
    for b in order:
        r = RESULTS[(sname, b)]
        rows.append([b] + [('-' if r[k] is None else f.format(r[k])) for k, _, f in cols])
    return head, rows


for sname in SKINS:
    head, rows = table(sname)
    print(f'### {sname} (h = {H})')
    print('| ' + ' | '.join(head) + ' |')
    print('|' + '---|' * len(head))
    for r in rows:
        print('| ' + ' | '.join(r) + ' |')
    print()

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
w = 0.8 / len(order)
for j, (key, title, log) in enumerate([('tets', 'tetrahedra', True), ('time', 'time (s, best of 3; adapt: 1 run)', True),
                                       ('sliver_sj', 'tets with scaled Jacobian < 0.2', False)]):
    for i, b in enumerate(order):
        vals = [max(RESULTS[(s, b)][key], 1e-3 if log else 0) for s in SKINS]
        ax[j].bar(np.arange(len(SKINS)) + i * w, vals, w, label=b)
    ax[j].set_xticks(np.arange(len(SKINS)) + 0.4 - w / 2)
    ax[j].set_xticklabels(list(SKINS))
    ax[j].set_title(title)
    if log:
        ax[j].set_yscale('log')
ax[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(len(SKINS), 3, figsize=(15, 3.6 * len(SKINS)))
for i, s in enumerate(SKINS):
    for b in order:
        r = RESULTS[(s, b)]
        axes[i][0].hist(r['sj'], bins=40, range=(0, 1), histtype='step', label=b, density=True)
        axes[i][1].hist(r['dmin_all'], bins=40, range=(0, 70), histtype='step', label=b, density=True)
        axes[i][2].hist(r['dmax_all'], bins=40, range=(70, 180), histtype='step', label=b, density=True)
    axes[i][0].set_ylabel(s)
for a, t in zip(axes[0], ['scaled Jacobian', 'smallest dihedral per tet (deg)', 'largest dihedral per tet (deg)']):
    a.set_title(t)
axes[0][0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## Cut-away views

The sphere's and the bunny's tetrahedra with the half beyond a plane removed, coloured by scaled Jacobian (red is poor). If GL is unavailable the cell falls back to a matplotlib scatter of the cell centroids on the cut plane.

In [ ]:
def to_grid(mesh):
    import os, tempfile
    path = os.path.join(tempfile.mkdtemp(), 'v.vtu')
    mp.write(path, mesh)
    return pv.read(path)


def cutaway(mesh, sj):
    cen = mesh.points[mesh.cells[0].data].mean(axis=1)
    mid = 0.5 * (mesh.points[:, 0].min() + mesh.points[:, 0].max())
    keep = cen[:, 0] < mid
    if GL:
        try:
            g = to_grid(mesh)
            g.cell_data['scaled_jacobian'] = sj
            sub = g.extract_cells(np.where(keep)[0])
            pl = pv.Plotter(off_screen=True, window_size=(500, 500))
            pl.add_mesh(sub, scalars='scaled_jacobian', clim=(0, 1), cmap='RdYlGn', show_edges=True,
                        show_scalar_bar=False, line_width=0.3)
            pl.camera_position = 'yz'
            pl.camera.azimuth = 30; pl.camera.elevation = 15
            img = pl.screenshot(return_img=True)
            pl.close()
            return img
        except Exception as exc:  # GL unavailable
            print('PyVista render skipped:', exc)
    sel = keep & (np.abs(cen[:, 0] - cen[keep][:, 0].max()) < 0.1 * np.ptp(cen[:, 0]))
    fig, a = plt.subplots(figsize=(5, 5))
    a.scatter(cen[sel, 1], cen[sel, 2], c=sj[sel], cmap='RdYlGn', vmin=0, vmax=1, s=8)
    a.set_aspect('equal'); a.axis('off')
    fig.canvas.draw()
    img = np.asarray(fig.canvas.buffer_rgba())[..., :3].copy()
    plt.close(fig)
    return img


fig, axes = plt.subplots(2, len(order), figsize=(4 * len(order), 8.6))
for row, sk in enumerate(('sphere', 'bunny')):
    for a, b in zip(axes[row], order):
        r = RESULTS[(sk, b)]
        a.imshow(cutaway(r['mesh'], r['sj']))
        a.set_title(f"{sk}: {b}\n{r['tets']} tets, SJ min {r['sj_min']:.2f}", fontsize=10)
        a.axis('off')
plt.tight_layout()
plt.show()

## What the adapted octree changes

`post_adapt` is the only difference between the two octree rows: the same stuffed lattice, then `adapt_volume`'s collapse, split, swap and smoothing towards the size field. The ratios below are adapted over plain.

In [ ]:
for s in SKINS:
    a, b = RESULTS[(s, 'octree')], RESULTS[(s, 'octree + adapt')]
    print(f"{s:9s} tets x{b['tets'] / a['tets']:.2f}, SJ<0.2 {a['sliver_sj']} -> {b['sliver_sj']}, "
          f"dihedral {a['dmin']:.1f}-{a['dmax']:.1f} -> {b['dmin']:.1f}-{b['dmax']:.1f}, "
          f"SJ mean {a['sj_mean']:.3f} -> {b['sj_mean']:.3f}")

## Reading the comparison

* **Skin kept or fitted.** Netgen, TetGen and Gmsh keep the input triangles (the core checks it), so neighbouring regions and CAD faces conform; the octree backends fit their own boundary within the reported distance, and the adapted mesh re-measures that distance after remeshing.
* **Count against quality.** The tables put the tetrahedra count beside the angle range, the sliver counts and the scaled Jacobian and aspect-ratio statistics: a backend with fewer elements is not better unless its worst elements are acceptable.
* **Time** is the wall clock of the whole call (best of three; the adapted octree is timed once, as its remeshing pass is by far the slowest step) (the external programs include writing and reading their files).
* TetGen and Gmsh appear only when their programs are installed; Netgen only when the core or the `meshioplusplus-netgen` add-on provides it.